## TBA-based OPR + cOPR logger

This notebook pulls match results and component titles from The Blue Alliance and recomputes every team's total OPR plus component OPRs (cOPRs) using only matches that have finished so far. The output JSON follows the OSF_epa_data(2).json structure (event / matches / teams), but each team entry now contains the freshly computed OPR/cOPR values for that match's timestamp.

You must set the TBA_AUTH_KEY environment variable (https://www.thebluealliance.com/account) before running this notebook to authorize the requests.


In [4]:
import json
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import requests

TBA_BASE_URL = "https://www.thebluealliance.com/api/v3"
TBA_AUTH_KEY = "uqTThWSrIgK7D7M3ct9fnwfIrj9m7ZzuCjwsgWsHzMtRl2xRNIm8pEQXVhfwOsBv"
if not TBA_AUTH_KEY:
    raise RuntimeError("Set the TBA_AUTH_KEY environment variable (https://www.thebluealliance.com/account) before running this notebook.")

SESSION = requests.Session()
SESSION.headers.update({
    "X-TBA-Auth-Key": TBA_AUTH_KEY,
    "Accept": "application/json",
})

REQUEST_DELAY = 0.2


In [5]:
def fetch_tba_json(path: str, params: Dict[str, Any] | None = None) -> Dict[str, Any]:
    response = SESSION.get(f"{TBA_BASE_URL}{path}", params=params, timeout=10)
    time.sleep(REQUEST_DELAY)
    response.raise_for_status()
    return response.json()


def match_sort_key(match: Dict[str, Any]) -> tuple:
    timestamp = match.get("actual_time") or match.get("time") or 0
    comp_level = match.get("comp_level", "").lower()
    return (
        timestamp,
        {"qm": 0, "ef": 1, "qf": 2, "sf": 3, "f": 4}.get(comp_level, 99),
        match.get("set_number") or 0,
        match.get("match_number") or 0,
    )


def parse_team_number(team_key: Any) -> int:
    if isinstance(team_key, str) and team_key.startswith("frc"):
        return int(team_key[3:])
    return int(team_key)


def enrich_match(match: Dict[str, Any]) -> None:
    for color in ("red", "blue"):
        team_keys = match["alliances"][color]["team_keys"]
        match["alliances"][color]["team_numbers"] = [parse_team_number(team_key) for team_key in team_keys]

def completed_matches(matches: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    filtered = []
    for match in matches:
        breakdown = match.get("score_breakdown")
        alliances = match.get("alliances")
        if not breakdown or not alliances:
            continue
        if alliances["red"].get("score") is None or alliances["blue"].get("score") is None:
            continue
        filtered.append(match)
    return filtered

def infer_components(coprs_snapshot: Dict[str, Any], matches: List[Dict[str, Any]]) -> List[str]:
    scoreboard_keys = set()
    for match in matches:
        breakdown = match.get("score_breakdown", {})
        for color in ("red", "blue"):
            for component, value in breakdown.get(color, {}).items():
                if isinstance(value, (int, float)):
                    scoreboard_keys.add(component)
    scoreboard_keys = sorted(scoreboard_keys)
    if coprs_snapshot:
        first = next(iter(coprs_snapshot.values()))
        ordered = [component for component in first.keys() if component in scoreboard_keys]
        return ordered or scoreboard_keys
    return scoreboard_keys

def extract_component_value(match: Dict[str, Any], component: str, color: str) -> float:
    breakdown = match.get("score_breakdown", {})
    value = breakdown.get(color, {}).get(component)
    return float(value) if isinstance(value, (int, float)) else 0.0

def build_system_matrix(
    matches: List[Dict[str, Any]], team_index: Dict[int, int], component: str | None = None
) -> tuple[np.ndarray, np.ndarray]:
    rows = []
    totals = []
    for match in matches:
        for color in ("red", "blue"):
            row = np.zeros(len(team_index), dtype=float)
            for team_number in match["alliances"][color]["team_numbers"]:
                row[team_index[team_number]] = 1.0
            rows.append(row)
            if component is None:
                score = match["alliances"][color].get("score")
                totals.append(float(score) if score is not None else 0.0)
            else:
                totals.append(extract_component_value(match, component, color))
    if not rows:
        return np.zeros((0, len(team_index)), dtype=float), np.zeros(0, dtype=float)
    return np.vstack(rows), np.array(totals, dtype=float)

def solve_least_squares(
    matrix: np.ndarray, vector: np.ndarray, team_count: int
) -> np.ndarray:
    if matrix.size == 0:
        return np.zeros(team_count, dtype=float)
    solution, *_ = np.linalg.lstsq(matrix, vector, rcond=None)
    return solution.ravel()

def extract_event_opr_data(event_key: str, output_file: Path) -> Dict[str, Any]:
    matches = fetch_tba_json(f"/event/{event_key}/matches")
    coprs_snapshot = fetch_tba_json(f"/event/{event_key}/coprs")
    matches = sorted(completed_matches(matches), key=match_sort_key)
    components = infer_components(coprs_snapshot, matches)

    team_index: Dict[int, int] = {}
    team_order: List[int] = []
    processed_matches: List[Dict[str, Any]] = []
    match_records: List[Dict[str, Any]] = []

    def ensure_team(team_number: int) -> None:
        if team_number not in team_index:
            team_index[team_number] = len(team_order)
            team_order.append(team_number)

    for match in matches:
        enrich_match(match)
        for color in ("red", "blue"):
            for team_number in match["alliances"][color]["team_numbers"]:
                ensure_team(team_number)
        processed_matches.append(match)
        total_matrix, total_vector = build_system_matrix(processed_matches, team_index)
        total_opr = solve_least_squares(total_matrix, total_vector, len(team_order))
        component_oprs: Dict[str, np.ndarray] = {}
        for component in components:
            _, comp_vector = build_system_matrix(processed_matches, team_index, component)
            component_oprs[component] = solve_least_squares(total_matrix, comp_vector, len(team_order))

        team_snapshots = []
        for color in ("red", "blue"):
            for team_number in match["alliances"][color]["team_numbers"]:
                idx = team_index[team_number]
                copr_values = {
                    component: float(component_oprs[component][idx]) for component in components
                }
                team_snapshots.append(
                    {
                        "team": team_number,
                        "alliance": color,
                        "opr": float(total_opr[idx]),
                        "copr": copr_values,
                    }
                )

        match_records.append(
            {
                "match_key": match["key"],
                "time": match.get("time"),
                "status": match.get("status"),
                "teams": team_snapshots,
            }
        )

    result = {
        "event": event_key,
        "extracted_at": datetime.utcnow().isoformat() + "Z",
        "components": components,
        "matches": match_records,
    }

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with output_file.open("w", encoding="utf-8") as writer:
        json.dump(result, writer, indent=2)

    return result


In [6]:
event_key = "2026orore"
output_path = Path("2026orore_opr_data.json")

event_data = extract_event_opr_data(event_key, output_path)
print(f"Computed OPR/cOPR for {len(event_data['matches']):,} matches and wrote {output_path}")


Computed OPR/cOPR for 52 matches and wrote 2026orore_opr_data.json


C:\Users\Aarush\AppData\Local\Temp\ipykernel_23312\2984252776.py:146: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "extracted_at": datetime.utcnow().isoformat() + "Z",
